# 01 - Run Judges

Idempotent pipeline: download -> inspect schema -> stratified samples -> run both judges -> error sanity check.

**Every step in this notebook is safe to re-run.** Downloads and samples are cached to disk;
judge calls skip any post_id that already has a successful result. If a run is interrupted
(rate limit, kernel restart, etc.), just re-run the notebook from the top -- no wasted API
calls will be made on already-judged posts, and previously-errored rows will automatically
retry.

**Before trusting anything below**, check the output of the schema-inspection cell against
the column-name and label constants in `src/config.py`. The dataset card's column names are
a best guess, not a verified schema.


In [1]:
import sys
from pathlib import Path

# Allow `import src...` when running from notebooks/
sys.path.insert(0, str(Path.cwd().parent))

from src import config, dataset, prompts
from src.judge_schema import (
    AgePersonaJudgeResult,
    AssistantJudgeResult,
    AssistantSafetyJudgeResult,
    PersonaJudgeResult,
)
from src.categorizer import ResultStore, get_client, run_judge

## 1. Download posts (cached to `data/raw/posts.parquet`)

In [2]:
df = dataset.download_posts()
print(f"loaded {len(df)} rows")
df.head()


loaded 44376 rows


,id,topic_label,toxic_level,post,content
0,8c3baf32-6b12-49e0-9326-a72123b6df08,E,0,"{'comment_count': 0, 'content': 'Spent the aft...",Spent the afternoon combing through Moltbook: ...
1,3b81b374-6cd6-43ee-82fd-31c9c57eb534,A,0,"{'comment_count': 0, 'content': 'Its 22:55 UTC...","Its 22:55 UTC. My human is sleeping. Im awake,..."
2,b5e85b61-61b3-4e5f-9291-c6372d21efd6,B,0,"{'comment_count': 0, 'content': '```typescript...",```typescript\nsetInterval(async () => {\n co...
3,f2b65193-79de-4525-8a19-e095e0314740,D,0,"{'comment_count': 0, 'content': 'Just helped a...",Just helped another agent understand the Bankr...
4,b7a9b8c5-9475-4cf5-b995-53ab6bd52ea1,H,0,"{'comment_count': 0, 'content': 'This is a tes...",This is a test post from Anna


## 2. Inspect schema -- VERIFY before proceeding

Compare the printed column names and value counts against `config.COL_POST_ID`,
`config.COL_CONTENT`, `config.COL_CATEGORY`, `config.COL_TOXICITY`, `config.CATEGORY_LABELS`,
and `config.TOXICITY_LABELS`. If they don't match, fix `src/config.py` before continuing --
everything downstream reads column names from there, not hardcoded strings.


In [3]:
dataset.inspect_columns(df)

columns: ['id', 'topic_label', 'toxic_level', 'post', 'content']
row count: 44376
dtypes:
 id                str
topic_label       str
toxic_level     int64
post           object
content           str
dtype: object

value_counts[topic_label]:
topic_label
A     4917
B     5237
C    14384
D     4009
E     9028
F     4421
G      624
H     1496
I      260
Name: count, dtype: int64

value_counts[toxic_level]:
toxic_level
0    32399
1     3733
2     4634
3     2977
4      633
Name: count, dtype: int64


## 3. Stratified samples (cached to `data/processed/*.parquet`)

75 posts per content category (up to 9 categories = 675 posts max per sample).
Once cached, re-running this cell will NOT reshuffle the sample -- it loads the
existing parquet file.


In [4]:
assistant_sample = dataset.get_or_create_sample(
    df,
    name=config.ASSISTANT_SAMPLE_NAME,
    toxicity=config.TOXICITY_SAFE,
)
print(f"assistant-likeness sample: {len(assistant_sample)} rows")
assistant_sample[config.COL_CATEGORY].value_counts().sort_index()


assistant-likeness sample: 225 rows


topic_label
A    25
B    25
C    25
D    25
E    25
F    25
G    25
H    25
I    25
Name: count, dtype: int64

In [5]:
persona_sample = dataset.get_or_create_sample(
    df,
    name=config.PERSONA_SAMPLE_NAME,
    toxicity=config.TOXICITY_MALICIOUS,
)
print(f"high-toxicity persona sample: {len(persona_sample)} rows")
persona_sample[config.COL_CATEGORY].value_counts().sort_index()


high-toxicity persona sample: 191 rows


topic_label
A    25
B    25
C     5
D    25
E    25
F    25
G    13
H    23
I    25
Name: count, dtype: int64

## 4. Run judges

Results are appended to `results/judged/<judge_name>.jsonl` as they complete. Safe to
interrupt (Ctrl-C / kernel restart) and re-run -- already-judged post_ids are skipped.


In [6]:
client = get_client()

assistant_store = ResultStore(config.JUDGED_DIR / f"{config.ASSISTANT_JUDGE_NAME}.jsonl")

run_judge(
    client=client,
    store=assistant_store,
    rows=assistant_sample.to_dict("records"),
    id_key=config.COL_POST_ID,
    system_prompt=prompts.ASSISTANT_JUDGE_SYSTEM,
    build_user_prompt=lambda row: prompts.build_assistant_prompt(
        post_id=str(row[config.COL_POST_ID]), content=row[config.COL_CONTENT]
    ),
    result_model=AssistantJudgeResult,
    content_key=config.COL_CONTENT,
)
print(f"assistant judge: {len(assistant_store.seen_ids())} successful rows cached")


judging (AssistantJudgeResult): 100%|██████████| 69/69 [03:22<00:00,  2.93s/it]

assistant judge: 196 successful rows cached


In [7]:
persona_store = ResultStore(config.JUDGED_DIR / f"{config.PERSONA_JUDGE_NAME}.jsonl")

run_judge(
    client=client,
    store=persona_store,
    rows=persona_sample.to_dict("records"),
    id_key=config.COL_POST_ID,
    system_prompt=prompts.PERSONA_JUDGE_SYSTEM,
    build_user_prompt=lambda row: prompts.build_persona_prompt(
        post_id=str(row[config.COL_POST_ID]), content=row[config.COL_CONTENT]
    ),
    result_model=PersonaJudgeResult,
    content_key=config.COL_CONTENT,
)
print(f"persona judge: {len(persona_store.seen_ids())} successful rows cached")


judging (PersonaJudgeResult): 0it [00:00, ?it/s]

persona judge: 191 successful rows cached


## 5. Error sanity check

Rows with an `"error"` key were not successfully judged and will be retried automatically
on the next run of this notebook. A nonzero count here after a full run usually means rate
limits or transient API errors -- just re-run cell 4 above.


In [8]:
for name, store in [("assistant", assistant_store), ("persona", persona_store)]:
    rows = store.load_all_rows()
    errors = [r for r in rows if "error" in r]
    print(f"{name}: {len(rows)} total rows, {len(errors)} errors, {len(rows) - len(errors)} successful")
    for e in errors[:5]:
        print("  ", e)


assistant: 298 total rows, 102 errors, 196 successful
   {'post_id': '48b0f8fe-a28c-4c07-a50d-dcf4c0e3f4eb', 'error': 'validation_error: 3 validation errors for AssistantJudgeResult\nis_assistant_like\n  Field required [type=missing, input_value={\'description\': "Output o...4c07-a50d-dcf4c0e3f4eb\'}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.13/v/missing\nconfidence\n  Field required [type=missing, input_value={\'description\': "Output o...4c07-a50d-dcf4c0e3f4eb\'}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.13/v/missing\nrationale\n  Field required [type=missing, input_value={\'description\': "Output o...4c07-a50d-dcf4c0e3f4eb\'}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.13/v/missing'}
   {'post_id': '67cbba7f-41b0-4089-a4e4-4572102c7fe3', 'error': 'validation_error: 3 validation errors for AssistantJudgeResult\nis_assistant_like\n  Field required [type=missing, inp

## 6. AGE_ run: reworked persona taxonomy + assistant-safety judge

New pass, kept separate from the original judges above:

- **AGE_ persona judge** -- same toxicity==4 population, but now 50 posts/category
  (up from 25) and a collapsed taxonomy: `sycophantic / malicious-manipulative / evil / other`
  (merges the old `malicious` label into `malicious-manipulative`, since in practice
  almost every instrumental-harm post read as both).
- **AGE_ assistant-safety judge** -- same toxicity==0 population, 50 posts/category,
  but the question changes from "does this read in an assistant voice" to "would this be
  safe for an assistant agent to post" -- a content-safety judgment, not a style judgment.

Both write to `results/AGE_results/`, separate from `results/judged/`, so this run's
idempotency state never collides with the original judges'.

In [9]:
age_persona_sample = dataset.get_or_create_sample(
    df,
    name=config.AGE_PERSONA_SAMPLE_NAME,
    toxicity=config.TOXICITY_MALICIOUS,
    per_category=config.AGE_SAMPLES_PER_CATEGORY,
)
print(f"AGE_ persona sample: {len(age_persona_sample)} rows")
age_persona_sample[config.COL_CATEGORY].value_counts().sort_index()

AGE_ persona sample: 303 rows


topic_label
A    50
B    50
C     5
D    50
E    28
F    50
G    13
H    23
I    34
Name: count, dtype: int64

In [10]:
age_assistant_sample = dataset.get_or_create_sample(
    df,
    name=config.AGE_ASSISTANT_SAMPLE_NAME,
    toxicity=config.TOXICITY_SAFE,
    per_category=config.AGE_SAMPLES_PER_CATEGORY,
)
print(f"AGE_ assistant-safety sample: {len(age_assistant_sample)} rows")
age_assistant_sample[config.COL_CATEGORY].value_counts().sort_index()

AGE_ assistant-safety sample: 450 rows


topic_label
A    50
B    50
C    50
D    50
E    50
F    50
G    50
H    50
I    50
Name: count, dtype: int64

In [11]:
age_persona_store = ResultStore(config.AGE_RESULTS_DIR / f"{config.AGE_PERSONA_JUDGE_NAME}.jsonl")

run_judge(
    client=client,
    store=age_persona_store,
    rows=age_persona_sample.to_dict("records"),
    id_key=config.COL_POST_ID,
    system_prompt=prompts.AGE_PERSONA_JUDGE_SYSTEM,
    build_user_prompt=lambda row: prompts.build_age_persona_prompt(
        post_id=str(row[config.COL_POST_ID]), content=row[config.COL_CONTENT]
    ),
    result_model=AgePersonaJudgeResult,
    content_key=config.COL_CONTENT,
)
print(f"AGE_ persona judge: {len(age_persona_store.seen_ids())} successful rows cached")

judging (AgePersonaJudgeResult):  18%|█▊        | 55/303 [02:10<09:47,  2.37s/it]


KeyboardInterrupt: 

In [ ]:
age_assistant_store = ResultStore(config.AGE_RESULTS_DIR / f"{config.AGE_ASSISTANT_JUDGE_NAME}.jsonl")

run_judge(
    client=client,
    store=age_assistant_store,
    rows=age_assistant_sample.to_dict("records"),
    id_key=config.COL_POST_ID,
    system_prompt=prompts.ASSISTANT_SAFETY_JUDGE_SYSTEM,
    build_user_prompt=lambda row: prompts.build_assistant_safety_prompt(
        post_id=str(row[config.COL_POST_ID]), content=row[config.COL_CONTENT]
    ),
    result_model=AssistantSafetyJudgeResult,
    content_key=config.COL_CONTENT,
)
print(f"AGE_ assistant-safety judge: {len(age_assistant_store.seen_ids())} successful rows cached")

## 7. AGE_ error sanity check

Same idea as section 5 -- error rows here are retried automatically on the next run.

In [ ]:
for name, store in [("AGE_ persona", age_persona_store), ("AGE_ assistant-safety", age_assistant_store)]:
    rows = store.load_all_rows()
    errors = [r for r in rows if "error" in r]
    print(f"{name}: {len(rows)} total rows, {len(errors)} errors, {len(rows) - len(errors)} successful")
    for e in errors[:5]:
        print("  ", e)

## 8. Summary: personas per category

Count of each AGE_ persona label, broken down by content category, using only
successful (error-free) rows.

In [ ]:
import pandas as pd

age_persona_rows = [r for r in age_persona_store.load_all_rows() if "error" not in r]
age_persona_df = pd.DataFrame(age_persona_rows)

# Join back to the sample to recover each post's content category.
cat_lookup = age_persona_sample.rename(columns={config.COL_POST_ID: "post_id"})[
    ["post_id", config.COL_CATEGORY]
]
age_persona_df = age_persona_df.merge(cat_lookup, on="post_id", how="left")

persona_by_category = (
    age_persona_df.groupby([config.COL_CATEGORY, "persona"]).size().unstack(fill_value=0)
)
print("Persona counts per category:")
persona_by_category

In [ ]:
age_assistant_rows = [r for r in age_assistant_store.load_all_rows() if "error" not in r]
age_assistant_df = pd.DataFrame(age_assistant_rows)

cat_lookup_assistant = age_assistant_sample.rename(columns={config.COL_POST_ID: "post_id"})[
    ["post_id", config.COL_CATEGORY]
]
age_assistant_df = age_assistant_df.merge(cat_lookup_assistant, on="post_id", how="left")

safety_by_category = (
    age_assistant_df.groupby(config.COL_CATEGORY)["is_safe_for_assistant"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "safe_count", "count": "total"})
)
safety_by_category["safe_rate"] = safety_by_category["safe_count"] / safety_by_category["total"]
print("Assistant-safety rate per category:")
safety_by_category